# Model 4a: Delta Team Qualities → Delta Player Qualities — Midfielders## Cambio de enfoque: ahora predecimos el CAMBIOEn los modelos 1–3 predecíamos el **nivel** post-transfer (`to_Qi`). Ahora predecimos el **cambio** (`delta_Qi`).$$\Delta PQ_i = \alpha + \sum_{k=1}^{7} \gamma_k \cdot \Delta TQ_k$$Donde:- $\Delta PQ_i = \text{to}_{Q_i} - \text{from}_{Q_i}$ → cuánto cambió la quality del jugador- $\Delta TQ_k = \text{to\_team}_{k} - \text{from\_team\_proj}_{k}$ → cuánto cambia el estilo tácticoEn español:> **Cambio del jugador = f( cambio de estilo táctico )**Solo 7 features. Pregunta simple: ¿el cambio de entorno táctico por sí solo explica cuánto cambia un jugador?---

In [ ]:
import pandas as pdimport numpy as npimport statsmodels.api as smfrom sklearn.metrics import r2_score, mean_absolute_error, mean_squared_errorimport plotly.graph_objects as goimport plotly.express as pxfrom plotly.subplots import make_subplotsimport warningswarnings.filterwarnings("ignore")pd.set_option("display.max_columns", 40)pd.set_option("display.float_format", "{:.4f}".format)DATA = "../../../../thesis_data/processed_data/thesis_model_dataset/active/within_league_transfers_v5.parquet"df = pd.read_parquet(DATA)mf = df[df["from_position"] == "Midfielder"].copy()print(f"Midfielders: {len(mf):,} rows")

In [ ]:
QUALITIES = [    "Active defence", "Aerial threat", "Box threat", "Composure",    "Defensive heading", "Dribbling", "Effectiveness", "Finishing",    "Hold-up play", "Intelligent defence", "Involvement",    "Passing quality", "Pressing", "Progression",    "Providing teammates", "Run quality", "Winning duels",]TEAM_Q = ["DEFENCE", "DEFENSIVE_TRANSITION", "ATTACKING_TRANSITION",          "ATTACK", "PENETRATION", "CHANCE_CREATION", "OUTCOME"]TEAM_Q_LABELS = {    "DEFENCE": "Defence", "DEFENSIVE_TRANSITION": "Def. Transition",    "ATTACKING_TRANSITION": "Att. Transition", "ATTACK": "Attack",    "PENETRATION": "Penetration", "CHANCE_CREATION": "Chance Creation",    "OUTCOME": "Outcome",}from_pq = [f"from_{q}" for q in QUALITIES]to_pq   = [f"to_{q}" for q in QUALITIES]from_tq = [f"from_q_proj_{q}" for q in TEAM_Q]to_tq   = [f"to_q_{q}" for q in TEAM_Q]# Compute deltasfor q in QUALITIES:    mf[f"delta_{q}"] = mf[f"to_{q}"] - mf[f"from_{q}"]for q in TEAM_Q:    mf[f"delta_team_{q}"] = mf[f"to_q_{q}"] - mf[f"from_q_proj_{q}"]mf["delta_Minutes"] = mf["to_Minutes"] - mf["from_Minutes"]delta_pq = [f"delta_{q}" for q in QUALITIES]delta_tq = [f"delta_team_{q}" for q in TEAM_Q]

## Visualización: Distribución de DeltasCambios en team qualities y player qualities across all transfers.

In [ ]:
fig = make_subplots(rows=1, cols=2, subplot_titles=["Delta Team Qualities", "Delta Player Qualities (sample)"])for q in TEAM_Q:    fig.add_trace(go.Box(y=mf[f"delta_team_{q}"].dropna(), name=TEAM_Q_LABELS[q], boxmean=True), row=1, col=1)sample_pq = ["Pressing", "Involvement", "Passing quality", "Dribbling", "Finishing"]for q in sample_pq:    fig.add_trace(go.Box(y=mf[f"delta_{q}"].dropna(), name=q, boxmean=True), row=1, col=2)fig.update_layout(height=450, template="plotly_white", showlegend=False,    title="Distribución de deltas (inputs y outputs)")fig.show()

---## Resultados7 delta team qualities → predice cada `delta_Qi` por separado (17 modelos OLS).

In [ ]:
features = delta_tq# Clean + splitkeep = features + delta_pq + ["from_season"]clean = mf[keep].dropna()train = clean[clean["from_season"] <= 2023]test  = clean[clean["from_season"] == 2024]print(f"Clean: {len(clean):,} | Train: {len(train):,} | Test: {len(test):,}")print(f"Features: {len(features)}")results = []models = {}for q in QUALITIES:    target = f"delta_{q}"    X_train = sm.add_constant(train[features])    X_test  = sm.add_constant(test[features])    y_train = train[target]    y_test  = test[target]    model = sm.OLS(y_train, X_train).fit()    y_pred = model.predict(X_test)    results.append({        "Quality": q,        "R²_train": model.rsquared,        "R²_adj": model.rsquared_adj,        "R²_test": r2_score(y_test, y_pred),        "MAE_test": mean_absolute_error(y_test, y_pred),        "F_pvalue": model.f_pvalue,    })    models[q] = modelres = pd.DataFrame(results)print(res.to_string(index=False))print(f"\nMean R²_train: {res['R²_train'].mean():.4f}")print(f"Mean R²_adj:   {res['R²_adj'].mean():.4f}")print(f"Mean R²_test:  {res['R²_test'].mean():.4f}")print(f"Mean MAE_test: {res['MAE_test'].mean():.4f}")

### Comparación con modelos anteriores

In [ ]:
# Recompute M3 (from_pq + from_tq + to_tq -> to_Qi) for comparisonm3_features = from_pq + from_tq + to_tqm3_keep = m3_features + to_pq + ["from_season"]m3_clean = mf[m3_keep].dropna()m3_train = m3_clean[m3_clean["from_season"] <= 2023]m3_test  = m3_clean[m3_clean["from_season"] == 2024]m3_r2 = []for q in QUALITIES:    X_tr = sm.add_constant(m3_train[m3_features])    X_te = sm.add_constant(m3_test[m3_features])    m = sm.OLS(m3_train[f"to_{q}"], X_tr).fit()    m3_r2.append(r2_score(m3_test[f"to_{q}"], m.predict(X_te)))comp = pd.DataFrame({"Quality": QUALITIES, "R²_M3_levels": m3_r2})comp["R²_M4a_delta_tq"] = res["R²_test"].valuesprint(comp.to_string(index=False))print(f"\nMean R²_test per model:")for col in [c for c in comp.columns if c.startswith("R²")]:    print(f"  {col}: {comp[col].mean():.4f}")

In [ ]:
# Bar chart comparisonfig = go.Figure()r2_cols = [c for c in comp.columns if c.startswith("R²")]colors = ["#636EFA", "#EF553B", "#00CC96", "#AB63FA", "#FFA15A", "#19D3F3"]for i, col in enumerate(r2_cols):    fig.add_trace(go.Bar(name=col.replace("R²_", ""), x=comp["Quality"], y=comp[col],        marker_color=colors[i % len(colors)]))fig.update_layout(title="R²_test — Comparación de modelos (Midfielders)",    yaxis_title="R²_test", barmode="group", height=450, template="plotly_white",    xaxis_tickangle=-45)fig.show()

### Full OLS Summary — Best and Worst

In [ ]:
best_q = res.loc[res["R²_test"].idxmax(), "Quality"]worst_q = res.loc[res["R²_test"].idxmin(), "Quality"]print(f"BEST: {best_q} (R²_test = {res.loc[res['R²_test'].idxmax(), 'R²_test']:.4f})")print(models[best_q].summary())print(f"\n{'='*70}")print(f"WORST: {worst_q} (R²_test = {res.loc[res['R²_test'].idxmin(), 'R²_test']:.4f})")print(models[worst_q].summary())

---## TakeawaySolo con deltas de team qualities el modelo explica muy poco del cambio del jugador (R²_test ~3-5%). El cambio de estilo táctico por sí solo **no es suficiente**.Esto tiene sentido: si un jugador con Pressing = 0.8 se mueve a un equipo que presiona igual, su delta depende mucho más de regression to the mean (su nivel previo) que del contexto táctico.**Siguiente modelo (4b):** Agregar las pre-transfer qualities como features — saber *de dónde parte* el jugador además de *cómo cambia el entorno*.